<a href="https://colab.research.google.com/github/krjeyy/AI-ML-lab-experiments/blob/main/random_forest%2Csvm%2CID3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#random forest
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data = load_iris()
X = data.data
y = data.target

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate model
print("Random Forest Accuracy:",
      accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=data.target_names))


In [ ]:
#svm
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data = load_iris()
X = data.data
y = data.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create a pipeline with feature scaling and SVM
model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1.0, gamma="scale")
)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate model
print("SVM Accuracy:",
      accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=data.target_names))


In [ ]:
#ID3
import numpy as np
from collections import Counter
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier

# Load dataset
data = load_iris()
X = data.data
y = data.target

# ID3 implementation for discrete-valued features
def entropy(y):
    counts = np.bincount(y)
    probabilities = counts[counts > 0] / len(y)
    return -np.sum(probabilities * np.log2(probabilities))

def information_gain(X_column, y):
    parent_entropy = entropy(y)
    values, counts = np.unique(X_column, return_counts=True)

    weighted_entropy = 0.0
    for value, count in zip(values, counts):
        subset_y = y[X_column == value]
        weighted_entropy += (count / len(y)) * entropy(subset_y)

    return parent_entropy - weighted_entropy

class ID3:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y, depth=0)
        return self

    def _build_tree(self, X, y, depth):
        # Stop if all samples have the same class
        if len(np.unique(y)) == 1:
            return y[0]

        # Stop if no features remain or depth limit is reached
        if X.shape[1] == 0 or (
            self.max_depth is not None and depth >= self.max_depth
        ):
            return Counter(y).most_common(1)[0][0]

        # Select the feature with highest information gain
        gains = [
            information_gain(X[:, i], y)
            for i in range(X.shape[1])
        ]
        best_feature = int(np.argmax(gains))

        # Stop if no useful split is available
        if gains[best_feature] <= 0:
            return Counter(y).most_common(1)[0][0]

        node = {
            "feature": best_feature,
            "majority": Counter(y).most_common(1)[0][0],
            "children": {}
        }

        # Split on each distinct value of the selected feature
        for value in np.unique(X[:, best_feature]):
            mask = X[:, best_feature] == value
            X_subset = np.delete(X[mask], best_feature, axis=1)
            y_subset = y[mask]

            node["children"][value] = self._build_tree(
                X_subset, y_subset, depth + 1
            )

        return node

    def _predict_one(self, row, tree):
        if not isinstance(tree, dict):
            return tree

        value = row[tree["feature"]]

        if value not in tree["children"]:
            return tree["majority"]

        # Remove the used feature before following the branch
        remaining = np.delete(row, tree["feature"])
        return self._predict_one(
            remaining, tree["children"][value]
        )

    def predict(self, X):
        return np.array([
            self._predict_one(row.copy(), self.tree)
            for row in X
        ])

# ID3 requires discrete features.
# Discretize Iris measurements into three bins.
X_discrete = np.column_stack([
    np.digitize(X[:, i],
                bins=np.histogram(X[:, i], bins=3)[1][1:-1])
    for i in range(X.shape[1])
])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_discrete, y, test_size=0.2, random_state=42
)

# Train ID3
model = ID3(max_depth=10)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("ID3 Accuracy:",
      accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=data.target_names))
